In [ ]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "stage1-nested"   # the branch with the code this notebook needs (set back to "main" after merging)

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", url, CODE], check=True)
# fetch + checkout + pull, so a clone left on another branch earlier in the session switches over
subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", CODE, "pull", "-q", "origin", BRANCH], check=True)
print(BRANCH, subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
_MINE = {"fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic",
         "stage1_structural", "nested_builder"}
for _m in list(sys.modules):
    if _m in _MINE or _m == "src" or _m.startswith("src."):   # src.* = the team's nested code
        sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

Mounted at /content/drive
stage1-nested 9a30690 updated



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ammara-asif125 (kashaf-saleem09-institute-of-business-administration) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs', 'checkpoints': '/content/drive/MyDrive/fyp_nested_cl/checkpoints'}
GPU: Tesla T4


In [ ]:
import numpy as np, pandas as pd

def table(configs):
    """One row per config: accuracy metrics, cost, and per-level means over tasks (nested arms)."""
    rows = []
    for cfg in configs:
        base = {"arm": sr.arm_label(cfg), "lr": cfg["lr"], "seed": cfg["seed"]}
        r = load_result(cfg, paths)
        if r is None:
            rows.append({**base, "status": "not finished"}); continue
        m = r["metrics"]
        row = {**base, "run_id": r["run_id"], "avg_incremental_acc": m["avg_incremental_acc"],
               "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
               # mean of the diagonal: accuracy on each task right after learning it (plasticity check)
               "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
               "steps": m["optimizer_steps"], "n_params": m["n_params"],
               "minutes": r["wall_clock_s"] / 60, "peak_gpu_mb": m.get("peak_gpu_mem_mb")}
        log = m.get("level_log_per_exp") or []
        for k in (log[0] if log else {}):
            if k.startswith(("updates/", "rel_displacement/", "deployed_rel_displacement/", "penalty_ratio/")):
                row[k] = float(np.mean([x[k] for x in log]))
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
EPOCHS = 10
SEEDS = [0, 1, 2, 3, 4]               # 0-2 exist and are skipped
ARMS = {"flat": 0.1, "nested_a": 0.5, "nested_b": 0.6}   # tuned lrs from validation (unchanged)
GATE1 = [sr.arm_cfg(name, stage="stage1_nested_control", epochs=EPOCHS, seed=s, lr=lr,
                    eval_split="test", save_checkpoints=True)
         for s in SEEDS for name, lr in ARMS.items()]   # seed outer: each seed finishes every arm first
for c in GATE1: print(run_id(c), sr.arm_label(c), c["lr"], c["seed"], "| done" if load_result(c, paths) else "")

07a97dd12ec3 flat_nl 0.1 0 | done
82f66588c5f1 nested_a 0.5 0 | done
ea0983312db5 nested_b 0.6 0 | done
78e133183087 flat_nl 0.1 1 | done
1a6e0b7da780 nested_a 0.5 1 | done
48523f3dfeea nested_b 0.6 1 | done
b09e3cc29bda flat_nl 0.1 2 | done
9da44b28d332 nested_a 0.5 2 | done
badf7f571542 nested_b 0.6 2 | done
97e56eaf8f25 flat_nl 0.1 3 | done
fee1f9d57dba nested_a 0.5 3 | done
3faea1c21912 nested_b 0.6 3 | done
a1e14e58bff7 flat_nl 0.1 4 | done
909b2256d515 nested_a 0.5 4 | done
f6544537e625 nested_b 0.6 4 | done


In [ ]:
out = run_sweep(GATE1, sr.run_experiment, paths)

[1/15] 07a97dd12ec3 already finished, skipping
[2/15] 82f66588c5f1 already finished, skipping
[3/15] ea0983312db5 already finished, skipping
[4/15] 78e133183087 already finished, skipping
[5/15] 1a6e0b7da780 already finished, skipping
[6/15] 48523f3dfeea already finished, skipping
[7/15] b09e3cc29bda already finished, skipping
[8/15] 9da44b28d332 already finished, skipping
[9/15] badf7f571542 already finished, skipping
[10/15] 97e56eaf8f25 already finished, skipping
[11/15] fee1f9d57dba already finished, skipping
[12/15] 3faea1c21912 already finished, skipping
[13/15] a1e14e58bff7 already finished, skipping
[14/15] 909b2256d515 already finished, skipping
[15/15] f6544537e625 already finished, skipping


In [ ]:
cdf = table(GATE1)
done = cdf[cdf["status"] != "not finished"] if "status" in cdf else cdf
assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
cols = ["avg_incremental_acc", "acc_just_learned", "final_avg_acc", "avg_forgetting", "minutes"]
display(done.groupby(["arm", "lr"])[cols].agg(["mean", "std", "count"]).round(2))

avg_incremental_acc             acc_just_learned              \
                            mean   std count             mean   std count   
arm      lr                                                                 
flat_nl  0.1               14.32  0.32     5            82.65  0.75     5   
nested_a 0.5               14.19  0.30     5            83.43  0.40     5   
nested_b 0.6               14.27  0.38     5            82.59  1.41     5   

             final_avg_acc             avg_forgetting             minutes  \
                      mean   std count           mean   std count    mean   
arm      lr                                                                 
flat_nl  0.1          4.29  0.27     5          82.49  0.67     5    8.78   
nested_a 0.5          4.36  0.20     5          83.24  0.42     5    8.63   
nested_b 0.6          4.28  0.32     5          82.42  1.45     5    8.55   

                          
               std count  
arm      lr               
flat_nl  0.1  0.17     5  
nested_a 0.5  0.13     5  
nested_b 0.6  0.16     5

In [6]:
AVG_GRID = [0.1, 0.3, 0.5, 0.7, 0.9]   # same grid as A
AVG = dict(nesting=dict(periods=[4, 4, 4]))
AVG_VAL = [sr.arm_cfg("nested_a", stage="stage1_nested_lr_grid", epochs=EPOCHS, seed=0, lr=lr, eval_split="val", **AVG)
           for lr in AVG_GRID]
for c in AVG_VAL: print(run_id(c), sr.arm_label(c), c["lr"])
out = run_sweep(AVG_VAL, sr.run_experiment, paths)
g = table(AVG_VAL)
display(g[["arm", "lr", "avg_incremental_acc", "acc_just_learned", "final_avg_acc", "minutes"]])
best = g.loc[g["avg_incremental_acc"].idxmax()]
print(f"best lr: {best['lr']} (avg incremental {best['avg_incremental_acc']:.2f})")
if best["lr"] in (min(AVG_GRID), max(AVG_GRID)): print("WARNING: best lr at the edge of the grid; extend AVG_GRID first.")

2524a11579fd nested_a_4-4-4 0.1
c00cc9899579 nested_a_4-4-4 0.3
28bd0efeafb9 nested_a_4-4-4 0.5
edca6115f73c nested_a_4-4-4 0.7
11a113638352 nested_a_4-4-4 0.9
[1/5] 2524a11579fd running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 74.00%  current 74.00%  train-loss 1.286
  exp 2/20: seen-avg 43.40%  current 86.80%  train-loss 1.034
  exp 3/20: seen-avg 21.20%  current 63.60%  train-loss 1.414
  exp 4/20: seen-avg 20.50%  current 82.00%  train-loss 1.143
  exp 5/20: seen-avg 12.96%  current 64.80%  train-loss 1.506
  exp 6/20: seen-avg 10.80%  current 64.80%  train-loss 1.426
  exp 7/20: seen-avg 11.89%  current 83.20%  train-loss 1.192
  exp 8/20: seen-avg 9.75%  current 78.00%  train-loss 1.119
  exp 9/20: seen-avg 7.78%  current 70.00%  train-loss 1.292
  exp 10/20: seen-avg 8.00%  current 80.00%  train-loss 1.214
  exp 11/20: seen-avg 6.25%  current 68.80%  train-loss 1.241
  exp 12/20: seen-avg 6.10%  current 73.20%  train-loss 1.301
  exp 13/20: seen-avg 4.92%  current 64.00%  train-loss 1.518
  exp 14/20: seen-avg 5.63%  current 78.80%  train-loss 1.379
  exp 15/20: seen-avg 5.31%  current 79.60%  train-loss 1.365
  exp 16/20: seen-avg 4.88%  current 78.00%  train-loss 1.282
  exp 17/2

acc_current_exp,▄█▁▇▁▁▇▅▃▆▃▄▁▆▆▅█▅▅▆
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


[2/5] c00cc9899579 running


...

/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 69.20%  current 69.20%  train-loss 1.270
  exp 2/20: seen-avg 44.00%  current 88.00%  train-loss 0.778
  exp 3/20: seen-avg 25.33%  current 76.00%  train-loss 1.098
  exp 4/20: seen-avg 20.80%  current 83.20%  train-loss 0.756
  exp 5/20: seen-avg 14.80%  current 74.00%  train-loss 1.041
  exp 6/20: seen-avg 12.13%  current 72.80%  train-loss 1.008
  exp 7/20: seen-avg 11.66%  current 81.60%  train-loss 0.910
  exp 8/20: seen-avg 10.65%  current 85.20%  train-loss 0.674
  exp 9/20: seen-avg 8.89%  current 80.00%  train-loss 0.847
  exp 10/20: seen-avg 8.60%  current 86.00%  train-loss 0.802
  exp 11/20: seen-avg 7.45%  current 82.00%  train-loss 0.766
  exp 12/20: seen-avg 6.53%  current 78.40%  train-loss 0.933
  exp 13/20: seen-avg 5.91%  current 76.80%  train-loss 1.076
  exp 14/20: seen-avg 6.29%  current 88.00%  train-loss 0.686
  exp 15/20: seen-avg 5.60%  current 84.00%  train-loss 0.832
  exp 16/20: seen-avg 5.27%  current 84.40%  train-loss 0.702
  exp 17/

acc_current_exp,▁█▄▆▃▂▆▇▅▇▆▄▄█▇▇█▆▅▆
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


,arm,lr,avg_incremental_acc,acc_just_learned,final_avg_acc,minutes
0,nested_a_4-4-4,0.1,13.538618,75.50,4.00,8.123188
1,nested_a_4-4-4,0.3,14.321072,81.40,4.08,7.633325
2,nested_a_4-4-4,0.5,14.298743,81.88,4.42,7.832379
3,nested_a_4-4-4,0.7,13.933479,81.10,3.94,7.908487
4,nested_a_4-4-4,0.9,14.061235,81.14,4.10,8.126749


best lr: 0.3 (avg incremental 14.32)


In [7]:
AVG_LR = 0.3                      # <- fill in from the grid above
if AVG_LR is None:
    print("set AVG_LR first")
else:
    AVG_TEST = [sr.arm_cfg("nested_a", stage="stage1_nested_control", epochs=EPOCHS, seed=s, lr=AVG_LR,
                           eval_split="test", save_checkpoints=True, **AVG) for s in SEEDS]
    for c in AVG_TEST: print(run_id(c), sr.arm_label(c), c["lr"], c["seed"])
    out = run_sweep(AVG_TEST, sr.run_experiment, paths)
    cdf = table(GATE1 + AVG_TEST)
    done = cdf[cdf["status"] != "not finished"] if "status" in cdf else cdf
    assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
    display(done.groupby(["arm", "lr"])[cols].agg(["mean", "std", "count"]).round(2))
    lvl = [c for c in done.columns if c.startswith(("updates/", "rel_displacement/"))]
    display(done.groupby(["arm", "lr"])[lvl].mean().T.round(4))

608fbb5f814b nested_a_4-4-4 0.3 0
5f8d3c518a7c nested_a_4-4-4 0.3 1
1c4ea3be5456 nested_a_4-4-4 0.3 2
aebb8bbd0802 nested_a_4-4-4 0.3 3
bab2c20b985e nested_a_4-4-4 0.3 4
[1/5] 608fbb5f814b running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 73.20%  current 73.20%  train-loss 0.989
  exp 2/20: seen-avg 44.70%  current 89.40%  train-loss 0.668
  exp 3/20: seen-avg 23.13%  current 69.40%  train-loss 0.989
  exp 4/20: seen-avg 20.35%  current 81.40%  train-loss 0.685
  exp 5/20: seen-avg 15.08%  current 75.40%  train-loss 0.962
  exp 6/20: seen-avg 12.40%  current 74.40%  train-loss 0.888
  exp 7/20: seen-avg 12.20%  current 85.40%  train-loss 0.768
  exp 8/20: seen-avg 10.72%  current 85.80%  train-loss 0.624
  exp 9/20: seen-avg 8.38%  current 75.40%  train-loss 0.774
  exp 10/20: seen-avg 8.10%  current 81.00%  train-loss 0.761
  exp 11/20: seen-avg 7.67%  current 84.40%  train-loss 0.728
  exp 12/20: seen-avg 6.32%  current 75.80%  train-loss 0.827
  exp 13/20: seen-avg 6.03%  current 78.40%  train-loss 0.903
  exp 14/20: seen-avg 6.43%  current 90.00%  train-loss 0.681
  exp 15/20: seen-avg 5.45%  current 81.80%  train-loss 0.790
  exp 16/20: seen-avg 5.46%  current 87.40%  train-loss 0.688
  exp 17/

acc_current_exp,▂▇▁▅▃▃▆▆▃▅▆▃▄█▅▇██▅▇
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


[2/5] 5f8d3c518a7c running


...

/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/base.py:389: UserWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your strategy NestedStrategy.__init__ method still has some positional-only or positional-or-keyword arguments. Consider removing them. Offending arguments: ['self', 'learner', 'rule', 'criterion', 'reset_rule_per_exp']. This can be achieved by adding a * in the argument list of your __init__ method just after 'self'. More info: https://peps.python.org/pep-3102/#specification
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/avalanche/training/templates/ba

  exp 1/20: seen-avg 77.00%  current 77.00%  train-loss 0.868
  exp 2/20: seen-avg 40.30%  current 80.60%  train-loss 0.733
  exp 3/20: seen-avg 28.33%  current 85.00%  train-loss 0.727
  exp 4/20: seen-avg 21.50%  current 86.00%  train-loss 0.711
  exp 5/20: seen-avg 15.08%  current 75.40%  train-loss 0.904
  exp 6/20: seen-avg 13.57%  current 81.40%  train-loss 0.775
  exp 7/20: seen-avg 11.03%  current 77.20%  train-loss 0.946
  exp 8/20: seen-avg 10.97%  current 87.80%  train-loss 0.654
  exp 9/20: seen-avg 9.93%  current 89.40%  train-loss 0.576
  exp 10/20: seen-avg 8.90%  current 89.00%  train-loss 0.648
  exp 11/20: seen-avg 7.56%  current 83.20%  train-loss 0.711
  exp 12/20: seen-avg 7.35%  current 88.20%  train-loss 0.670
  exp 13/20: seen-avg 5.83%  current 75.80%  train-loss 0.916
  exp 14/20: seen-avg 5.49%  current 76.80%  train-loss 0.795
  exp 15/20: seen-avg 5.72%  current 85.80%  train-loss 0.728
  exp 16/20: seen-avg 5.03%  current 80.40%  train-loss 0.819
  exp 17/

acc_current_exp,▂▃▅▆▁▄▂▇█▇▅▇▁▂▆▃▃██▇
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+25,...


avg_incremental_acc             acc_just_learned        \
                                  mean   std count             mean   std   
arm            lr                                                           
flat_nl        0.1               14.32  0.32     5            82.65  0.75   
nested_a       0.5               14.19  0.30     5            83.43  0.40   
nested_a_4-4-4 0.3               14.14  0.38     5            82.30  0.68   
nested_b       0.6               14.27  0.38     5            82.59  1.41   

                         final_avg_acc             avg_forgetting              \
                   count          mean   std count           mean   std count   
arm            lr                                                               
flat_nl        0.1     5          4.29  0.27     5          82.49  0.67     5   
nested_a       0.5     5          4.36  0.20     5          83.24  0.42     5   
nested_a_4-4-4 0.3     5          4.32  0.14     5          82.09  0.62     5   
nested_b       0.6     5          4.28  0.32     5          82.42  1.45     5   

                   minutes              
                      mean   std count  
arm            lr                       
flat_nl        0.1    8.78  0.17     5  
nested_a       0.5    8.63  0.13     5  
nested_a_4-4-4 0.3    9.31  0.38     5  
nested_b       0.6    8.55  0.16     5

arm,flat_nl,nested_a,nested_a_4-4-4,nested_b
lr,0.1,0.5,0.3,0.6
updates/stem+layer1,790.0000,49.0000,197.0000,790.0000
rel_displacement/stem+layer1,0.0991,0.0496,0.0842,0.0573
updates/layer2+layer3,790.0000,197.0000,197.0000,790.0000
rel_displacement/layer2+layer3,0.0666,0.0741,0.0553,0.0820
updates/layer4+fc,790.0000,790.0000,197.0000,790.0000
rel_displacement/layer4+fc,0.0807,0.1586,0.0766,0.1700
